# YOLOE-11S Joint Defect Detection and Segmentation

Kaggle-ready pipeline for the seven-dataset small-defect collection.

This notebook:

1. Loads the attached `SmallDefectPreprocessing` notebook output.
2. Matches images, box labels, and raster masks.
3. Recreates the fixed dataset-and-size-stratified 70/15/15 split with seed 42.
4. Converts raster masks into YOLO instance-segmentation polygons.
5. Audits conversion quality and dataset counts.
6. Fine-tunes `yoloe-11s-seg.pt`.
7. Evaluates validation, overall test, and small/medium/large test subsets.
8. Calculates false positives per image and exports a summary CSV.

Kaggle settings: enable a GPU and Internet, then attach the `SmallDefectPreprocessing` notebook output as an input.


In [ ]:
!pip install -q -U "ultralytics==8.4.70" wandb pyyaml


In [ ]:
import os
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image
from ultralytics import YOLOE
from ultralytics.models.yolo.yoloe import YOLOEPESegTrainer

print("OpenCV:", cv2.__version__)


In [ ]:
# Experiment identity
RUN_NAME = "RunG_yoloe11s_seg_imgsz640"
MODEL_NAME = "yoloe-11s-seg.pt"
MODEL_LABEL = "YOLOE-11S-seg"
WANDB_PROJECT = "smallDefectDetection"

# Dataset structure
DATASET_NAMES = [
    "DAGM",
    "GC10-DET",
    "KolektorSDD2",
    "MPDD",
    "MTD",
    "Severstal",
    "VisA",
]
SIZE_BUCKETS = ["small", "medium", "large"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# Fixed split used by earlier runs
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

# Training settings kept aligned with the YOLO baselines
IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 100
PATIENCE = 25
WORKERS = 2
DEVICE = 0

# Polygon conversion settings
# CHAIN_APPROX_SIMPLE already removes redundant collinear points without an
# aggressive area filter. MIN_COMPONENT_AREA=1 preserves tiny defects.
MIN_COMPONENT_AREA = 1
MIN_POLYGON_POINTS = 3

KAGGLE_INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
PREPARED_DATASET_DIR = WORKING_ROOT / "yoloe_defect_dataset"
RUNS_DIR = WORKING_ROOT / "runs" / "segment"

random.seed(SEED)
np.random.seed(SEED)
os.environ["WANDB_PROJECT"] = WANDB_PROJECT


In [ ]:
# Optional W&B login. The notebook still runs if the secret is unavailable.
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return os.environ.get(name)


wandb_key = get_secret("wandb_api_key")
if wandb_key:
    import wandb
    wandb.login(key=wandb_key)
    print("W&B enabled. Project:", WANDB_PROJECT)
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("W&B secret not found; continuing with W&B disabled.")


## 1. Locate the attached preprocessing output


In [ ]:
expected_datasets = set(DATASET_NAMES)
valid_roots = []

print("Searching under:", KAGGLE_INPUT_ROOT)

for root, dirs, _ in os.walk(KAGGLE_INPUT_ROOT):
    matches = expected_datasets.intersection(dirs)
    if len(matches) >= 5:
        root_path = Path(root)
        valid_roots.append((len(matches), root_path))
        print("Candidate:", root_path)
        print("Matches:", sorted(matches))

if not valid_roots:
    raise FileNotFoundError(
        "Could not find the processed dataset. Attach the "
        "SmallDefectPreprocessing notebook output as a Kaggle input."
    )

# Prefer the candidate containing the largest number of expected datasets.
valid_roots.sort(key=lambda item: (-item[0], len(str(item[1]))))
SOURCE_ROOT = valid_roots[0][1]

print()
print("Using source root:", SOURCE_ROOT)
print("Available datasets:", sorted(p.name for p in SOURCE_ROOT.iterdir() if p.is_dir()))


## 2. Match images, masks, and existing box labels


In [ ]:
def index_files(directory, suffixes):
    return {
        path.stem: path
        for path in directory.iterdir()
        if path.is_file() and path.suffix.lower() in suffixes
    }


def candidate_stems(image_stem, target):
    base = image_stem.removesuffix("_defect")
    if target == "mask":
        return [
            image_stem,
            image_stem.replace("_defect", "_mask"),
            base,
            base + "_mask",
            base + "_gt",
        ]
    return [
        image_stem,
        image_stem.replace("_defect", "_bbs"),
        base,
        base + "_bbs",
    ]


samples = []
missing = []

for dataset_name in DATASET_NAMES:
    for size_bucket in SIZE_BUCKETS:
        bucket_root = SOURCE_ROOT / dataset_name / size_bucket
        image_dir = bucket_root / "images"
        mask_dir = bucket_root / "masks"
        box_dir = bucket_root / "labels_yolo"

        if not image_dir.exists() or not mask_dir.exists() or not box_dir.exists():
            missing.append((dataset_name, size_bucket, "missing directory"))
            continue

        mask_index = index_files(mask_dir, IMAGE_EXTS)
        box_index = index_files(box_dir, {".txt"})
        matched = 0

        # Preserve the same source iteration order used by the earlier runs so
        # seed 42 recreates their split assignment as closely as possible.
        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in IMAGE_EXTS:
                continue

            mask_path = next(
                (mask_index[s] for s in candidate_stems(image_path.stem, "mask") if s in mask_index),
                None,
            )
            box_path = next(
                (box_index[s] for s in candidate_stems(image_path.stem, "box") if s in box_index),
                None,
            )

            if mask_path is None or box_path is None:
                missing.append((dataset_name, size_bucket, image_path.name))
                continue

            samples.append({
                "image_path": image_path,
                "mask_path": mask_path,
                "box_path": box_path,
                "dataset": dataset_name,
                "size": size_bucket,
                "stratum": dataset_name + "_" + size_bucket,
            })
            matched += 1

        print(f"{dataset_name}/{size_bucket}: {matched} matched")

print()
print("Total matched samples:", len(samples))
print("Missing entries:", len(missing))
print("First missing entries:", missing[:10])

if not samples:
    raise RuntimeError("No image-mask-label triplets were matched.")

if len(samples) != 12670:
    print("WARNING: expected 12,670 samples but matched", len(samples))


## 3. Recreate the fixed stratified split


In [ ]:
by_stratum = defaultdict(list)
for sample in samples:
    by_stratum[sample["stratum"]].append(sample)

train_samples, val_samples, test_samples = [], [], []
rng = random.Random(SEED)

for stratum, group in sorted(by_stratum.items()):
    group = list(group)
    rng.shuffle(group)
    n = len(group)
    n_train = int(n * TRAIN_RATIO)
    n_val = int(n * VAL_RATIO)

    train_samples.extend(group[:n_train])
    val_samples.extend(group[n_train:n_train + n_val])
    test_samples.extend(group[n_train + n_val:])

rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)


def print_split_counts(name, split_samples):
    counts = Counter(sample["size"] for sample in split_samples)
    print(name, "total:", len(split_samples))
    print("  small:", counts["small"])
    print("  medium:", counts["medium"])
    print("  large:", counts["large"])


print_split_counts("Train", train_samples)
print_split_counts("Validation", val_samples)
print_split_counts("Test", test_samples)
print("Grand total:", len(train_samples) + len(val_samples) + len(test_samples))

assert train_samples and val_samples and test_samples
assert not ({id(s) for s in train_samples} & {id(s) for s in val_samples})
assert not ({id(s) for s in train_samples} & {id(s) for s in test_samples})


## 4. Convert raster masks into YOLO polygons


In [ ]:
def yolo_box_union_mask(label_path, height, width):
    union = np.zeros((height, width), dtype=np.uint8)
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cx, cy, bw, bh = map(float, parts[1:5])
        x1 = max(0, int(round((cx - bw / 2) * width)))
        y1 = max(0, int(round((cy - bh / 2) * height)))
        x2 = min(width, int(round((cx + bw / 2) * width)))
        y2 = min(height, int(round((cy + bh / 2) * height)))
        if x2 > x1 and y2 > y1:
            union[y1:y2, x1:x2] = 1
    return union


def bbox_iou_from_masks(mask_a, mask_b):
    def bounds(mask):
        ys, xs = np.where(mask > 0)
        if len(xs) == 0:
            return None
        return xs.min(), ys.min(), xs.max() + 1, ys.max() + 1

    a = bounds(mask_a)
    b = bounds(mask_b)
    if a is None or b is None:
        return 0.0

    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    intersection = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return intersection / max(area_a + area_b - intersection, 1)


def load_binary_mask(mask_path, box_path, expected_shape):
    gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f"Could not read mask: {mask_path}")

    height, width = expected_shape
    if gray.shape != (height, width):
        raise ValueError(
            f"Mask/image shape mismatch for {mask_path.name}: "
            f"mask={gray.shape}, image={(height, width)}"
        )

    # Otsu handles masks stored as 0/1, 0/255, or grayscale probability-like images.
    _, thresholded = cv2.threshold(gray, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    candidates = [thresholded.astype(np.uint8), (1 - thresholded).astype(np.uint8)]

    # Existing box labels tell us which polarity corresponds to the defect.
    box_mask = yolo_box_union_mask(box_path, height, width)
    scores = [bbox_iou_from_masks(candidate, box_mask) for candidate in candidates]
    binary = candidates[int(np.argmax(scores))]

    return binary


def mask_to_polygons(binary_mask):
    num_labels, component_map, stats, _ = cv2.connectedComponentsWithStats(
        binary_mask.astype(np.uint8), connectivity=8
    )
    polygons = []

    for component_id in range(1, num_labels):
        area = int(stats[component_id, cv2.CC_STAT_AREA])
        if area < MIN_COMPONENT_AREA:
            continue

        component = (component_map == component_id).astype(np.uint8)
        contours, _ = cv2.findContours(
            component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
        )
        if not contours:
            continue

        contour = max(contours, key=cv2.contourArea).reshape(-1, 2)
        unique_points = np.unique(contour, axis=0)

        # A one-pixel component cannot form an area polygon. Preserve it as a
        # one-pixel rectangle so tiny labelled defects are not silently dropped.
        if len(unique_points) < MIN_POLYGON_POINTS:
            x = int(stats[component_id, cv2.CC_STAT_LEFT])
            y = int(stats[component_id, cv2.CC_STAT_TOP])
            w = int(stats[component_id, cv2.CC_STAT_WIDTH])
            h = int(stats[component_id, cv2.CC_STAT_HEIGHT])
            x1 = max(0, x - 1)
            y1 = max(0, y - 1)
            x2 = min(binary_mask.shape[1] - 1, x + max(w, 1))
            y2 = min(binary_mask.shape[0] - 1, y + max(h, 1))
            contour = np.array([
                [x1, y1],
                [x2, y1],
                [x2, y2],
                [x1, y2],
            ], dtype=np.int32)

        polygons.append(contour)

    return polygons


def polygon_lines_and_reconstruction(binary_mask):
    height, width = binary_mask.shape
    reconstructed = np.zeros_like(binary_mask)
    lines = []

    for polygon in mask_to_polygons(binary_mask):
        polygon = polygon.copy()
        polygon[:, 0] = np.clip(polygon[:, 0], 0, width - 1)
        polygon[:, 1] = np.clip(polygon[:, 1], 0, height - 1)
        cv2.fillPoly(reconstructed, [polygon.astype(np.int32)], 1)

        normalized = polygon.astype(np.float64)
        normalized[:, 0] /= max(width - 1, 1)
        normalized[:, 1] /= max(height - 1, 1)
        coordinates = " ".join(f"{value:.8f}" for value in normalized.reshape(-1))
        lines.append("0 " + coordinates)

    return lines, reconstructed


def mask_metrics(original, reconstructed):
    intersection = np.logical_and(original, reconstructed).sum()
    union = np.logical_or(original, reconstructed).sum()
    denominator = original.sum() + reconstructed.sum()
    iou = intersection / union if union else 1.0
    dice = 2 * intersection / denominator if denominator else 1.0
    area_ratio = reconstructed.sum() / original.sum() if original.sum() else np.nan
    return float(iou), float(dice), float(area_ratio)


In [ ]:
def reset_directory(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


reset_directory(PREPARED_DATASET_DIR)

export_groups = {
    "train": train_samples,
    "val": val_samples,
    "test": test_samples,
    "test_small": [s for s in test_samples if s["size"] == "small"],
    "test_medium": [s for s in test_samples if s["size"] == "medium"],
    "test_large": [s for s in test_samples if s["size"] == "large"],
}

for split_name in export_groups:
    (PREPARED_DATASET_DIR / "images" / split_name).mkdir(parents=True, exist_ok=True)
    (PREPARED_DATASET_DIR / "labels" / split_name).mkdir(parents=True, exist_ok=True)

audit_rows = []
conversion_failures = []

for split_name, split_samples in export_groups.items():
    print("Exporting", split_name, len(split_samples))

    for index, sample in enumerate(split_samples):
        source_image = sample["image_path"]
        safe_name = (
            f"{sample['dataset']}_{sample['size']}_{index:06d}_{source_image.name}"
        )
        destination_image = PREPARED_DATASET_DIR / "images" / split_name / safe_name
        destination_label = (
            PREPARED_DATASET_DIR / "labels" / split_name / (Path(safe_name).stem + ".txt")
        )

        try:
            with Image.open(source_image) as image:
                width, height = image.size

            binary_mask = load_binary_mask(
                sample["mask_path"], sample["box_path"], (height, width)
            )
            lines, reconstructed = polygon_lines_and_reconstruction(binary_mask)

            if not lines:
                raise ValueError("No valid polygon produced")

            iou, dice, area_ratio = mask_metrics(binary_mask, reconstructed)
            shutil.copy2(source_image, destination_image)
            destination_label.write_text("\n".join(lines))

            audit_rows.append({
                "split": split_name,
                "dataset": sample["dataset"],
                "size": sample["size"],
                "image": safe_name,
                "instances": len(lines),
                "mask_pixels": int(binary_mask.sum()),
                "polygon_pixels": int(reconstructed.sum()),
                "iou": iou,
                "dice": dice,
                "area_ratio": area_ratio,
            })
        except Exception as error:
            conversion_failures.append({
                "split": split_name,
                "image": str(source_image),
                "error": str(error),
            })

audit_df = pd.DataFrame(audit_rows)
failures_df = pd.DataFrame(conversion_failures)
audit_df.to_csv(WORKING_ROOT / "mask_polygon_conversion_audit.csv", index=False)
failures_df.to_csv(WORKING_ROOT / "mask_polygon_conversion_failures.csv", index=False)

print()
print("Converted samples:", len(audit_df))
print("Conversion failures:", len(failures_df))
if len(failures_df):
    display(failures_df.head(20))

assert len(failures_df) == 0, "Fix conversion failures before training."


## 5. Audit polygon conversion and exported counts


In [ ]:
conversion_summary = audit_df.groupby(["split", "size"])[
    ["iou", "dice", "area_ratio", "instances"]
].agg(["count", "mean", "median", "min"])

display(conversion_summary)

for split_name, expected_samples in export_groups.items():
    image_count = sum(
        1 for path in (PREPARED_DATASET_DIR / "images" / split_name).iterdir()
        if path.suffix.lower() in IMAGE_EXTS
    )
    label_count = len(list((PREPARED_DATASET_DIR / "labels" / split_name).glob("*.txt")))
    print(split_name, "images:", image_count, "labels:", label_count)
    assert image_count == len(expected_samples)
    assert label_count == len(expected_samples)

print()
print("Lowest-IoU conversions:")
display(audit_df.sort_values("iou").head(20))


In [ ]:
# Visual verification: lowest-IoU example from each size group.
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for row_index, size_name in enumerate(SIZE_BUCKETS):
    audit_row = audit_df[
        (audit_df["split"] == "train") & (audit_df["size"] == size_name)
    ].sort_values("iou").iloc[0]

    exported_image = PREPARED_DATASET_DIR / "images" / "train" / audit_row["image"]
    exported_label = PREPARED_DATASET_DIR / "labels" / "train" / (Path(audit_row["image"]).stem + ".txt")

    # Locate the corresponding source sample through its safe filename suffix.
    source_sample = next(
        sample for sample in train_samples
        if audit_row["image"].endswith(sample["image_path"].name)
        and audit_row["dataset"] == sample["dataset"]
        and audit_row["size"] == sample["size"]
    )

    image = cv2.cvtColor(cv2.imread(str(exported_image)), cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    original = load_binary_mask(
        source_sample["mask_path"], source_sample["box_path"], (height, width)
    )
    reconstructed = np.zeros_like(original)

    for line in exported_label.read_text().splitlines():
        values = np.array([float(v) for v in line.split()[1:]], dtype=np.float64).reshape(-1, 2)
        values[:, 0] *= max(width - 1, 1)
        values[:, 1] *= max(height - 1, 1)
        cv2.fillPoly(reconstructed, [np.rint(values).astype(np.int32)], 1)

    comparison = np.zeros((height, width, 3), dtype=np.uint8)
    comparison[..., 1] = original * 255
    comparison[..., 0] = reconstructed * 255

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(size_name + " image")
    axes[row_index, 1].imshow(original, cmap="gray")
    axes[row_index, 1].set_title("Original mask")
    axes[row_index, 2].imshow(reconstructed, cmap="gray")
    axes[row_index, 2].set_title("Polygon mask")
    axes[row_index, 3].imshow(comparison)
    axes[row_index, 3].set_title(f"Overlay IoU={audit_row['iou']:.3f}")

for axis in axes.flat:
    axis.axis("off")

plt.tight_layout()
plt.savefig(WORKING_ROOT / "mask_polygon_audit_examples.png", dpi=160, bbox_inches="tight")
plt.show()


## 6. Write training and size-specific evaluation YAML files


In [ ]:
def write_data_yaml(path, test_split):
    content = {
        "path": str(PREPARED_DATASET_DIR),
        "train": "images/train",
        "val": "images/val",
        "test": "images/" + test_split,
        "nc": 1,
        "names": ["defect"],
    }
    path.write_text(yaml.safe_dump(content, sort_keys=False))


data_yaml_all = WORKING_ROOT / "data_all.yaml"
data_yaml_small = WORKING_ROOT / "data_small.yaml"
data_yaml_medium = WORKING_ROOT / "data_medium.yaml"
data_yaml_large = WORKING_ROOT / "data_large.yaml"

write_data_yaml(data_yaml_all, "test")
write_data_yaml(data_yaml_small, "test_small")
write_data_yaml(data_yaml_medium, "test_medium")
write_data_yaml(data_yaml_large, "test_large")

print(data_yaml_all.read_text())


## 7. Fine-tune YOLOE-11S-seg


In [ ]:
model = YOLOE(MODEL_NAME)

train_results = model.train(
    data=str(data_yaml_all),
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    optimizer="auto",
    seed=SEED,
    deterministic=True,
    workers=WORKERS,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    pretrained=True,
    exist_ok=True,
    val=True,
    save=True,
    plots=True,
    trainer=YOLOEPESegTrainer,
)


## 8. Locate and load the saved best checkpoint


In [ ]:
expected_best = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

if expected_best.exists():
    best_model_path = expected_best
else:
    candidates = sorted(WORKING_ROOT.glob(f"**/{RUN_NAME}/weights/best.pt"))
    print("best.pt candidates:")
    for candidate in candidates:
        print(candidate)
    if not candidates:
        raise FileNotFoundError("Training finished without a discoverable best.pt checkpoint.")
    best_model_path = candidates[0]

best_model = YOLOE(str(best_model_path))
print("Using checkpoint:", best_model_path)


## 9. Validation and separate test evaluation


In [ ]:
val_metrics = best_model.val(
    data=str(data_yaml_all),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME + "_val",
    exist_ok=True,
)

print(val_metrics)


In [ ]:
test_sets = {
    "overall": data_yaml_all,
    "small": data_yaml_small,
    "medium": data_yaml_medium,
    "large": data_yaml_large,
}

test_results = {}

for test_name, yaml_path in test_sets.items():
    print()
    print("Running test evaluation:", test_name)

    metrics = best_model.val(
        data=str(yaml_path),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=RUN_NAME + "_test_" + test_name,
        exist_ok=True,
    )
    test_results[test_name] = metrics


## 10. False positives per image using box IoU matching


In [ ]:
def box_iou_xyxy(box1, box2):
    if len(box1) == 0 or len(box2) == 0:
        return torch.zeros((len(box1), len(box2)))

    x1 = torch.maximum(box1[:, None, 0], box2[None, :, 0])
    y1 = torch.maximum(box1[:, None, 1], box2[None, :, 1])
    x2 = torch.minimum(box1[:, None, 2], box2[None, :, 2])
    y2 = torch.minimum(box1[:, None, 3], box2[None, :, 3])
    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)

    area1 = (box1[:, 2] - box1[:, 0]) * (box1[:, 3] - box1[:, 1])
    area2 = (box2[:, 2] - box2[:, 0]) * (box2[:, 3] - box2[:, 1])
    union = area1[:, None] + area2[None, :] - intersection
    return intersection / union.clamp(min=1e-6)


def polygons_to_xyxy(label_path, image_width, image_height):
    boxes = []
    if not label_path.exists():
        return torch.zeros((0, 4), dtype=torch.float32)

    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        points = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
        xs = points[:, 0] * image_width
        ys = points[:, 1] * image_height
        boxes.append([xs.min(), ys.min(), xs.max(), ys.max()])

    return torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4))


def compute_fp_per_image(model, image_dir, label_dir, conf=0.25, match_iou=0.5):
    image_paths = sorted(
        path for path in image_dir.iterdir()
        if path.suffix.lower() in IMAGE_EXTS
    )
    total_fp = 0

    for image_path in image_paths:
        prediction = model.predict(
            source=str(image_path),
            imgsz=IMG_SIZE,
            conf=conf,
            iou=0.7,
            device=DEVICE,
            verbose=False,
        )[0]

        image_height, image_width = prediction.orig_shape
        pred_boxes = (
            prediction.boxes.xyxy.cpu()
            if prediction.boxes is not None and len(prediction.boxes)
            else torch.zeros((0, 4))
        )
        confidences = (
            prediction.boxes.conf.cpu().numpy()
            if prediction.boxes is not None and len(prediction.boxes)
            else np.array([])
        )
        gt_boxes = polygons_to_xyxy(
            label_dir / (image_path.stem + ".txt"), image_width, image_height
        )

        if len(pred_boxes) == 0:
            continue
        if len(gt_boxes) == 0:
            total_fp += len(pred_boxes)
            continue

        ious = box_iou_xyxy(pred_boxes, gt_boxes)
        matched_gt = set()

        for pred_index in np.argsort(-confidences):
            best_gt = int(torch.argmax(ious[pred_index]).item())
            best_iou = float(ious[pred_index, best_gt].item())
            if best_iou >= match_iou and best_gt not in matched_gt:
                matched_gt.add(best_gt)
            else:
                total_fp += 1

    image_count = len(image_paths)
    return {
        "total_images": image_count,
        "total_fp": int(total_fp),
        "fp_per_image": total_fp / image_count if image_count else 0.0,
    }


fp_results = {}
for size_name, folder_name in {
    "overall": "test",
    "small": "test_small",
    "medium": "test_medium",
    "large": "test_large",
}.items():
    result = compute_fp_per_image(
        best_model,
        PREPARED_DATASET_DIR / "images" / folder_name,
        PREPARED_DATASET_DIR / "labels" / folder_name,
    )
    fp_results[size_name] = result
    print(size_name, result)


## 11. Export the final metrics table


In [ ]:
def metric_values(metric_group):
    return {
        "map50": float(metric_group.map50),
        "map50_95": float(metric_group.map),
        "precision": float(metric_group.mp),
        "recall": float(metric_group.mr),
    }


overall_box = metric_values(test_results["overall"].box)
overall_mask = metric_values(test_results["overall"].seg)
small_box = metric_values(test_results["small"].box)
medium_box = metric_values(test_results["medium"].box)
large_box = metric_values(test_results["large"].box)
small_mask = metric_values(test_results["small"].seg)
medium_mask = metric_values(test_results["medium"].seg)
large_mask = metric_values(test_results["large"].seg)

summary_row = {
    "Experiment": RUN_NAME,
    "Model": MODEL_LABEL,
    "Batch": BATCH_SIZE,
    "Epochs": EPOCHS,
    "Box_mAP50": overall_box["map50"],
    "Box_mAP50_95": overall_box["map50_95"],
    "Box_Precision": overall_box["precision"],
    "Box_Recall": overall_box["recall"],
    "Mask_mAP50": overall_mask["map50"],
    "Mask_mAP50_95": overall_mask["map50_95"],
    "Mask_Precision": overall_mask["precision"],
    "Mask_Recall": overall_mask["recall"],
    "Box_mAP50_Small": small_box["map50"],
    "Box_mAP50_Medium": medium_box["map50"],
    "Box_mAP50_Large": large_box["map50"],
    "Box_Recall_Small": small_box["recall"],
    "Box_Recall_Medium": medium_box["recall"],
    "Box_Recall_Large": large_box["recall"],
    "Mask_mAP50_Small": small_mask["map50"],
    "Mask_mAP50_Medium": medium_mask["map50"],
    "Mask_mAP50_Large": large_mask["map50"],
    "Mask_Recall_Small": small_mask["recall"],
    "Mask_Recall_Medium": medium_mask["recall"],
    "Mask_Recall_Large": large_mask["recall"],
    "Inference_Time_ms": float(test_results["overall"].speed.get("inference", np.nan)),
    "FP_Per_Image": fp_results["overall"]["fp_per_image"],
    "Parameters": int(sum(parameter.numel() for parameter in best_model.model.parameters())),
    "Checkpoint": str(best_model_path),
    "Notes": "ICCV 2025 YOLOE instance-segmentation baseline, imgsz 640",
}

summary_df = pd.DataFrame([summary_row])
summary_path = WORKING_ROOT / (RUN_NAME + "_summary.csv")
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print("Saved:", summary_path)


In [ ]:
# Confirm the artifacts that Kaggle Save Version should preserve.
required_outputs = [
    best_model_path,
    WORKING_ROOT / (RUN_NAME + "_summary.csv"),
    WORKING_ROOT / "mask_polygon_conversion_audit.csv",
    WORKING_ROOT / "mask_polygon_audit_examples.png",
]

for output_path in required_outputs:
    print(output_path, "exists:", output_path.exists())

assert all(path.exists() for path in required_outputs)
print("Pipeline completed successfully.")
